In [0]:
# Create environment widget
dbutils.widgets.dropdown("environment", "prod", ["dev", "preprod", "prod"], "Environment")

In [0]:
%run /Workspace/Users/lakshmisas96@gmail.com/orders-project/orders-analytics-pipeline/utils/config_loader

In [0]:
from pyspark.sql.functions import row_number, lit, current_timestamp, concat_ws, col
from pyspark.sql.window import Window

config = load_config_from_widget()
gold_schema = config['schemas']['gold']
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['catalog']}.{gold_schema}")

# DimCustomers
customers_path = config['storage']['landing']['customers']
df_dim_customers = (spark.read.parquet(customers_path)
    .withColumn("DimCustomerKey", row_number().over(Window.orderBy(lit(1))))
    .withColumn("customer_name", concat_ws(" ", col("first_name"), col("last_name")))
    .withColumn("created_date", current_timestamp())
    .select("DimCustomerKey", "customer_id", "customer_name", "email", "city", "state", "created_date")
)

external_path_customers = f"{config['storage']['external']['gold']}/DimCustomers"
df_dim_customers.write.format("delta").mode("overwrite").save(external_path_customers)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {config['catalog']}.{gold_schema}.DimCustomers
    USING DELTA LOCATION '{external_path_customers}'
""")

# DimProducts
products_path = config['storage']['landing']['products']
df_dim_products = (spark.read.parquet(products_path)
    .withColumn("DimProductKey", row_number().over(Window.orderBy(lit(1))))
    .withColumn("created_date", current_timestamp())
    .select("DimProductKey", "product_id", "product_name", "category", "brand", "price", "created_date")
)

external_path_products = f"{config['storage']['external']['gold']}/DimProducts"
df_dim_products.write.format("delta").mode("overwrite").save(external_path_products)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {config['catalog']}.{gold_schema}.DimProducts
    USING DELTA LOCATION '{external_path_products}'
""")

# DimRegion
region_path = config['storage']['landing']['region']
df_dim_region = (spark.read.parquet(region_path)
    .withColumn("DimRegionKey", row_number().over(Window.orderBy(lit(1))))
    .withColumn("created_date", current_timestamp())
    .select("DimRegionKey", "region_id", "region", "created_date")
)

external_path_region = f"{config['storage']['external']['gold']}/DimRegion"
df_dim_region.write.format("delta").mode("overwrite").save(external_path_region)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {config['catalog']}.{gold_schema}.DimRegion
    USING DELTA LOCATION '{external_path_region}'
""")

print("✅ All dimensions created!")
print(f"✅ DimCustomers: {external_path_customers}/_delta_log/")
print(f"✅ DimProducts: {external_path_products}/_delta_log/")
print(f"✅ DimRegion: {external_path_region}/_delta_log/")